# BARF — YOLO11n Fine-tune for Ping Pong Ball + Metal Ball → NCNN

Trains a 2-class YOLO11n detector — `ping_pong_ball` and `metal_ball` — then exports to NCNN format for the Android app.
The model detects **only** these two classes, nothing else.

**After training, labels in JS:**
```js
onDetection(function(frame) {
  var PING_PONG = 0, METAL = 1;
  var balls = frame.yolo.filter(d => (d.label === PING_PONG || d.label === METAL) && d.score > 0.5);
  if (balls.length > 0) move("forward", 0.5);
  else stop();
});
```

## Speed notes

This is tuned to get ~80% of the way there in a fraction of the wall-clock time of the original
100-epoch / 640px / full-backbone run:
- `imgsz=416` instead of 640 (training compute scales ~quadratically with resolution)
- `freeze=10` — freeze the pretrained backbone, only train the head/neck (fewer params to update per step)
- `epochs=40` with `patience=8` — early-stops well before 40 if validation mAP plateaus
- `cache="ram"` — decodes images once instead of every epoch
- Export shapes (640/320) for NCNN are unaffected by training `imgsz` — the exported graph is dynamic-shape regardless of what resolution it was trained at.

If you have more time and want to claw back the last bit of accuracy, bump `imgsz` to 640, drop `freeze`, and raise `epochs`.

## 1. Install dependencies

In [1]:
!pip install -U ultralytics roboflow pnnx ncnn

Defaulting to user installation because normal site-packages is not writeable


## 2. Download datasets from Roboflow

Two datasets, each single-class, merged below into one 2-class dataset.

In [2]:
from roboflow import Roboflow

rf = Roboflow(api_key="Mt0m4dCKTTlNf29OV3zm")

# Ping pong ball dataset (existing)
pingpong_project = rf.workspace("pingpong-ojuhj").project("ping-pong-detection-0guzq")
pingpong_version = pingpong_project.version(3)
pingpong_dataset = pingpong_version.download("yolov11")
print("Ping pong dataset:", pingpong_dataset.location)

# Metal balls dataset — https://universe.roboflow.com/unibots-fvalh/metal-balls
metal_project = rf.workspace("unibots-fvalh").project("metal-balls")
metal_version = metal_project.versions()[0]  # latest published version
metal_dataset = metal_version.download("yolov11")
print("Metal balls dataset:", metal_dataset.location)

loading Roboflow workspace...
loading Roboflow project...
Ping pong dataset: /home/voldemort/Documents/Coding_Projects/unibots/better-barf/BARF/tools/Ping-Pong-Detection-3
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 32.0%
Version export complete for yolov11 format



Extracting Dataset Version Zip to metal-balls-3 in yolov11:: 100%|██████████| 977/977 [00:00<00:00, 7864.30it/s]

Metal balls dataset: /home/voldemort/Documents/Coding_Projects/unibots/better-barf/BARF/tools/metal-balls-3


## 3. Merge into a single 2-class dataset

Forces every box from the ping pong dataset to class `0` (`ping_pong_ball`) and every box from the
metal balls dataset to class `1` (`metal_ball`), regardless of how the source datasets labeled them
internally — the merged model should only ever emit these two classes.

In [3]:
import os, shutil
from pathlib import Path

MERGED_DIR = Path("merged_dataset")
CLASS_NAMES = ["ping_pong_ball", "metal_ball"]

def merge_split(src_root, split, target_class_id, dst_root):
    src_images = Path(src_root) / split / "images"
    src_labels = Path(src_root) / split / "labels"
    if not src_images.exists():
        return 0
    dst_images = dst_root / split / "images"
    dst_labels = dst_root / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    n = 0
    for img_path in src_images.iterdir():
        if not img_path.is_file():
            continue
        # prefix filenames so the two source datasets can't collide
        dst_name = f"{Path(src_root).name}_{img_path.name}"
        dst_img_path = dst_images / dst_name
        if not dst_img_path.exists():
            try:
                os.symlink(img_path.resolve(), dst_img_path)
            except OSError:
                shutil.copy2(img_path, dst_img_path)

        label_path = src_labels / (img_path.stem + ".txt")
        dst_label_path = dst_labels / (dst_name.rsplit(".", 1)[0] + ".txt")
        lines = []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if not parts:
                    continue
                # remap whatever class the source used to our single target class
                parts[0] = str(target_class_id)
                lines.append(" ".join(parts))
        dst_label_path.write_text("\n".join(lines) + ("\n" if lines else ""))
        n += 1
    return n

if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)

counts = {}
for split in ["train", "valid", "test"]:
    n_pp = merge_split(pingpong_dataset.location, split, 0, MERGED_DIR)
    n_metal = merge_split(metal_dataset.location, split, 1, MERGED_DIR)
    counts[split] = (n_pp, n_metal)
    print(f"{split}: {n_pp} ping pong + {n_metal} metal ball images")

data_yaml = MERGED_DIR / "data.yaml"
data_yaml.write_text(
    f"train: train/images\n"
    f"val: valid/images\n"
    f"test: test/images\n"
    f"\n"
    f"nc: {len(CLASS_NAMES)}\n"
    f"names: {CLASS_NAMES}\n"
)
print("Wrote", data_yaml)

train: 22931 ping pong + 402 metal ball images
valid: 951 ping pong + 60 metal ball images
test: 956 ping pong + 24 metal ball images
Wrote merged_dataset/data.yaml


## 4. Train

In [4]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: False


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # nano — fastest on Snapdragon+Vulkan

def print_epoch_accuracy(trainer):
    m = trainer.metrics
    p = m.get("metrics/precision(B)", 0)
    r = m.get("metrics/recall(B)", 0)
    map50 = m.get("metrics/mAP50(B)", 0)
    print(f"  -> epoch {trainer.epoch + 1} accuracy: precision={p:.3f} recall={r:.3f} mAP50={map50:.3f}")

model.add_callback("on_fit_epoch_end", print_epoch_accuracy)

results = model.train(
    data=str(data_yaml.resolve()),
    epochs=40,           # was 100 — patience below will stop earlier if it plateaus
    imgsz=416,            # was 640 — ~2.4x less compute per image
    batch=16,
    patience=8,           # was 20 — stop sooner once mAP stalls
    freeze=10,            # freeze pretrained backbone, only train neck/head
    cache="ram",          # decode images once, not every epoch
    save=True,            # save last.pt/best.pt after every epoch (so a crash never loses progress)
    save_period=1,        # also keep a separate weights/epoch{N}.pt each epoch, in case last.pt is mid-write when things go wrong
    hsv_h=0.015,          # colour augment — helps with different lighting
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=10,           # slight rotation — balls are round so safe
    scale=0.5,            # random scale — helps detect balls at various distances
    mosaic=1.0,
    name="balls",
)

best_pt = "runs/detect/balls/weights/best.pt"
print("Best weights:", best_pt)

## 5. Validate

In [ ]:
from ultralytics import YOLO

best_pt = "runs/detect/balls/weights/best.pt"
model = YOLO(best_pt)
metrics = model.val(data=str(data_yaml.resolve()))

print(f"mAP50 (overall):    {metrics.box.map50:.3f}")
print(f"mAP50-95 (overall): {metrics.box.map:.3f}")
print(f"Precision:          {metrics.box.mp:.3f}")
print(f"Recall:             {metrics.box.mr:.3f}")

### What these numbers mean

- **mAP50** — out of all the balls in the validation set, how many did the model find with a "close
  enough" box (50% overlap)? Closer to **1.0 is better**. Above ~0.7 is solid for a small/fast model
  like this on a phone.
- **mAP50-95** — same idea but graded much more strictly on how tight the box is. Always lower than
  mAP50 — that's expected, not a bug.
- **Precision** — of the boxes the model drew, what fraction were actually a ball (vs. a false alarm)?
  Low precision = it's drawing boxes on things that aren't balls.
- **Recall** — of the real balls in frame, what fraction did it actually find? Low recall = it's
  missing balls that are there.

**Rule of thumb for this robot:** recall matters more than precision for "find the ball and drive
at it" — a missed ball means the robot does nothing, a false positive just means one bad frame of
steering, which the next frame usually corrects.

In [ ]:
names = metrics.names  # {0: 'ping_pong_ball', 1: 'metal_ball'}
print(f"{'class':<16} {'AP50':>8} {'precision':>10} {'recall':>8}")
for i, name in names.items():
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f"{name:<16} {ap50:>8.3f} {p:>10.3f} {r:>8.3f}")

## 6. Export to TorchScript
PNNX needs a TorchScript file as input.

In [ ]:
import shutil, os

best_pt = "runs/detect/balls/weights/best.pt"
!yolo export model={best_pt} format=torchscript

ts_src = best_pt.replace(".pt", ".torchscript")
ts_dst = "yolo11n_balls.torchscript"
shutil.copy2(ts_src, ts_dst)
print(f"Copied {ts_src} → {ts_dst}")

## 7. Initial PNNX conversion (static shape)

In [ ]:
!pnnx yolo11n_balls.torchscript

## 8. Patch PNNX script for dynamic shape
Required for NCNN to accept variable input sizes on the phone. These numbers correspond to the
fixed 640/320 export shapes used in step 9, independent of the training `imgsz` above.

In [ ]:
import os

def patch_yolo_script(filename):
    if not os.path.exists(filename):
        print(f"Not found: {filename}")
        return

    with open(filename) as f:
        lines = f.readlines()

    out = []
    for line in lines:
        # Fix Area Attention for dynamic input sizes
        if 'v_96 = v_95.view(1, 2, 128, 1024)' in line:
            line = line.replace('1024', '-1')
        if 'v_106 = v_105.view(1, 128, 32, 32)' in line:
            line = "        v_106 = v_105.view(1, 128, v_95.size(2), v_95.size(3))\n"
        if 'v_107 = v_99.reshape(1, 128, 32, 32)' in line:
            line = "        v_107 = v_99.reshape(1, 128, v_95.size(2), v_95.size(3))\n"

        # Dynamic reshape for detection head outputs
        if '.view(1, ' in line and any(x in line for x in ['6400','1600','400','16384','4096','1024']):
            line = line.replace('6400','-1').replace('1600','-1').replace('400','-1')
            line = line.replace('16384','-1').replace('4096','-1').replace('1024','-1')
            line = line.rstrip() + ".transpose(1, 2)\n"

        # Adjust cat axis after transpose
        if 'torch.cat' in line and 'dim=2' in line:
            line = line.replace('dim=2', 'dim=1')

        # Drop post-process, return raw detections
        if 'return' in line and 'v_' in line:
            line = "        return v_238\n"

        out.append(line)

    with open(filename, 'w') as f:
        f.writelines(out)
    print(f"Patched {filename}")

patch_yolo_script("yolo11n_balls_pnnx.py")

## 9. Re-export patched script → final NCNN conversion

In [ ]:
import importlib

try:
    mod = importlib.import_module("yolo11n_balls_pnnx")
    importlib.reload(mod)
    mod.export_torchscript()
    print("Re-exported patched script")
except Exception as e:
    print(f"Re-export failed: {e} — continuing with pnnx direct")

!pnnx yolo11n_balls_pnnx.py.pt inputshape=[1,3,640,640] inputshape2=[1,3,320,320]

## 10. Rename outputs and copy to Android assets

In [ ]:
import os, shutil
from pathlib import Path

# Rename pnnx output to final asset names
renames = [
    ("yolo11n_balls_pnnx.py.ncnn.param", "yolo11n.ncnn.param"),
    ("yolo11n_balls_pnnx.py.ncnn.bin",   "yolo11n.ncnn.bin"),
]
for src, dst in renames:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied {src} → {dst}")
    else:
        print(f"WARNING: {src} not found")

# Copy to Android assets — adjust path to your checkout
assets = Path("../android/src/main/assets")
if assets.exists():
    for name in ["yolo11n.ncnn.param", "yolo11n.ncnn.bin"]:
        if os.path.exists(name):
            shutil.copy2(name, assets / name)
            print(f"Deployed {name} → {assets}")
else:
    print(f"Assets dir not found at {assets} — copy files manually")

## 11. Update class names in Android app

The fine-tuned model has **2 classes: label `0` = ping_pong_ball, label `1` = metal_ball**.
The draw overlay in `yolo11_det.cpp` still uses the COCO 80-class array,
so bounding box labels will say `"person"` / `"bicycle"` instead of the real names.

Fix: in `android/src/main/jni/yolo11_det.cpp`, replace the `class_names` array:

```cpp
static const char* class_names[] = { "ping_pong_ball", "metal_ball" };
```

Then rebuild the APK.

---

**JS detection script after deployment:**
```js
var PING_PONG = 0;
var METAL = 1;

onDetection(function(frame) {
  var balls = frame.yolo.filter(function(d) {
    return (d.label === PING_PONG || d.label === METAL) && d.score > 0.5;
  });

  if (balls.length === 0) { stop(); return; }

  // Steer toward the largest ball (biggest bounding box area)
  balls.sort(function(a, b) { return (b.w * b.h) - (a.w * a.h); });
  var ball = balls[0];

  log("Ball at x=" + ball.x + " label=" + ball.label + " score=" + ball.score.toFixed(2));

  if (ball.x < 560)       rotate("left",  0.35);
  else if (ball.x > 720)  rotate("right", 0.35);
  else                     move("forward", 0.5);
});
```

## 12. Download results (Kaggle)

Kaggle notebooks don't have Colab's `files.download()`. Zip the files that actually matter
(NCNN export + best weights, not every per-epoch checkpoint) and use `FileLink` to get a
clickable download link in the notebook output — no need to wait for a "Save Version" commit.

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink

# Kaggle's writable output dir — files placed here survive past the kernel session
OUT_DIR = Path("/kaggle/working")

download_files = [
    "yolo11n.ncnn.param",
    "yolo11n.ncnn.bin",
    "runs/detect/balls/weights/best.pt",
]

bundle_dir = OUT_DIR / "barf_model_bundle"
bundle_dir.mkdir(parents=True, exist_ok=True)
for f in download_files:
    src = Path(f)
    if src.exists():
        shutil.copy2(src, bundle_dir / src.name)
    else:
        print(f"WARNING: {f} not found, skipping")

zip_path = OUT_DIR / "barf_model_bundle"
archive = shutil.make_archive(str(zip_path), "zip", root_dir=bundle_dir)
print(f"Bundled {len(list(bundle_dir.iterdir()))} files into {archive}")

FileLink(archive)